In [8]:
# importing libraries
import html
import json
import os
import re
from dotenv import load_dotenv
from deep_translator import GoogleTranslator
from googleapiclient.discovery import build
from langdetect import DetectorFactory, detect
import pandas as pd
from tqdm import tqdm

# Ensure reproducible language detection
DetectorFactory.seed = 0

# Load API key from .env 
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found — check your .env file exists and is loaded correctly")

In [9]:
# youtube object
youtube = build("youtube", "v3", developerKey=API_KEY)

# channel detail request
channel_handle = "@SundaySarthak"
response = (
    youtube.channels().list(part="snippet,contentDetails,statistics", forHandle=channel_handle).execute()
)

# Key information
item = response["items"][0]
print(f"Channel Name : {item['snippet']['title']}")
print(f"Channel ID : {item['id']}")
print(f"Subscribers : {item['statistics']['subscriberCount']}")

uploads_playlist_id = item["contentDetails"]["relatedPlaylists"]["uploads"]
print(f"Upload ID : {uploads_playlist_id}")

Channel Name : Sarthak Goswami
Channel ID : UC5fcjujOsqD-126Chn_BAuA
Subscribers : 1890000
Upload ID : UU5fcjujOsqD-126Chn_BAuA


In [ ]:
# extracting recent 50 videos ids
playlist_items = []
page_token = None
print("Fetching last 50 uploads from playlist ...")

while len(playlist_items) < 50:
    response = (
        youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=min(50, 50 - len(playlist_items)),
            pageToken=page_token
        ).execute()
    )

    for item in response.get("items", []):
        playlist_items.append({
            "video_id": item["contentDetails"]["videoId"],
            "title": item["snippet"]["title"],
        })

    page_token = response.get("nextPageToken")
    if not page_token:
        break

print(f"Collected videos ids title {len(playlist_items)} videos")

Fetching last 50 uploads from playlist ...
Collected videos ids 50 videos


In [ ]:
# Extracting  video metadata: stats + tags, publish date, category
video_ids = [v["video_id"] for v in playlist_items]

stats_response = (
    youtube.videos().list(part="statistics,snippet", id=",".join(video_ids)).execute()
)

stats_by_id = {item["id"]: item for item in stats_response.get("items", [])}

for video in playlist_items:
    stat_item = stats_by_id.get(video["video_id"], {})
    stats = stat_item.get("statistics", {})
    snippet = stat_item.get("snippet", {})

    video["views"] = int(stats.get("viewCount", 0))
    video["likes"] = int(stats.get("likeCount", 0))
    video["comments"] = int(stats.get("commentCount", 0))
    video["tags"] = snippet.get("tags", [])
    video["published_at"] = snippet.get("publishedAt", None)
    video["category_id"] = snippet.get("categoryId", None)
    video["description"] = snippet.get("description", "")

print(f"Videos metadata and stats for {len(playlist_items)} videos.")

Videos metadata and stats for 50 videos.


In [14]:
raw_comments = []
per_vid_cap = 150

# Sampling both "relevance" (top/most-liked comments) and "time" (most
# recent comments) per video, rather than just one order, to avoid bias
# toward only highly-upvoted comments — recent comments are more likely
# to include newer slang/toxicity patterns not yet reflected in top comments.
for video in tqdm(playlist_items, desc="Extracting Videos"):
    for sort_order in ["relevance", "time"]:
        c_page_token = None
        fetched_in_mode = 0

        while fetched_in_mode < per_vid_cap:
            try:
                c_response = (
                    youtube.commentThreads()
                    .list(
                        part="snippet",
                        videoId=video["video_id"],
                        order=sort_order,
                        maxResults=min(100, per_vid_cap - fetched_in_mode),
                        pageToken=c_page_token
                    ).execute()
                )

                items = c_response.get("items", [])
                if not items:
                    break

                for item in items:
                    top_comment = item["snippet"]["topLevelComment"]["snippet"]
                    raw_comments.append({
                        "video_id": video["video_id"],
                        "comment_id": item["snippet"]["topLevelComment"]["id"],
                        "raw_text": top_comment.get("textOriginal"),
                        "like_count": top_comment.get("likeCount", 0),
                        "reply_count": item["snippet"].get("totalReplyCount", 0),
                        "published_at": top_comment.get("publishedAt"),
                        "sampling_mode": sort_order,
                    })
                    fetched_in_mode += 1
                    if fetched_in_mode >= per_vid_cap:
                        break

                c_page_token = c_response.get("nextPageToken")
                if not c_page_token:
                    break

            except Exception as e:
                # Some videos have comments disabled — this is expected,
                # not a bug, so we log and move to the next video/mode
                # rather than letting one failure kill the whole scrape.
                print(f"Skipping {video['video_id']} ({sort_order}): {e}")
                break

Extracting Videos: 100%|██████████| 50/50 [00:43<00:00,  1.14it/s]


In [ ]:
# Saving raw comments data 

if raw_comments:
    os.makedirs("../data/raw", exist_ok=True)
    raw_df = pd.DataFrame(raw_comments)
    raw_df.to_csv("../data/raw/raw_comment.csv", index=False, encoding="utf-8")

    print(f"\nExtraction complete! Saved {len(raw_df)} raw comments across {len(playlist_items)} videos.")
    print(f"Sampling distribution:\n{raw_df['sampling_mode'].value_counts()}")
else:
    print("\nNo comments were extracted. Check the error warnings printed above.")


Extraction complete! Saved 14890 raw comments across 50 videos.
Sampling distribution:
sampling_mode
time         7447
relevance    7443
Name: count, dtype: int64


In [16]:
# Saving raw videos metadata 
video_df = pd.DataFrame(playlist_items)
video_df.to_csv("../data/raw/raw_videos.csv", index=False, encoding="utf-8")
print(f"Saved metadata for {len(video_df)} videos.")

Saved metadata for 50 videos.
